#### Uniform functional annotation (eggNOG-mapper)

- **Limitation addressed:** the main analysis only had gene-product names for the 5 in-house-annotated *Hydrophis*
genomes — the 7 outgroup species (from NCBI) had no consistent functional labels
    - **Fix:** run real **eggNOG-mapper** (diamond mode, eggNOG5 orthology database) on all **12** proteomes independently, giving every gene a label from the same orthology-based method — no lineage gets special treatment.



In [1]:
import pandas as pd, glob, collections, os
CAFE="/hpcfs/users/a1864358/sanders_lab/phylogenomics/cafe"; os.chdir(CAFE)
files = sorted(glob.glob("eggnog/out12/*/*.emapper.annotations"))
print(f"{len(files)} species annotated")
counts = {f.split('/')[-2]: sum(1 for l in open(f) if not l.startswith('#')) for f in files}
pd.Series(counts, name="genes_annotated").sort_values(ascending=False)

12 species annotated


hmaj       18921
nhel       18789
tele       18681
hcurw      18633
horn       18512
vberus     18511
hcure      18496
hcy        18451
caspera    18251
vkomodo    17808
ptext      16829
nscut      16370
Name: genes_annotated, dtype: int64

##### Roll up gene-level labels to orthogroup consensus
For each orthogroup, vote across ALL 12 species' member genes (using their `Preferred_name`, falling back to `Description`) and take the majority label.

In [2]:
genelabel = {}
for f in files:
    df = pd.read_csv(f, sep="\t", comment="#", header=None,
        names=["query","seed_ortholog","evalue","score","eggNOG_OGs","max_annot_lvl","COG_category",
               "Description","Preferred_name","GOs","EC","KEGG_ko","KEGG_Pathway","KEGG_Module",
               "KEGG_Reaction","KEGG_rclass","BRITE","KEGG_TC","CAZy","BiGG_Reaction","PFAMs"])
    for _, r in df.iterrows():
        label = r["Preferred_name"] if r["Preferred_name"] not in ("-", None) else (
                 r["Description"] if r["Description"] not in ("-", None) else None)
        if label: genelabel[r["query"]] = label
print("genes with an eggNOG label:", len(genelabel))

og = pd.read_csv("/hpcfs/users/a1864358/sanders_lab/phylogenomics/orthofinder/orthofinder_prot/primary_transcripts_lifted/OrthoFinder/Results_lifted/Orthogroups/Orthogroups.tsv", sep="\t").fillna("")
sp_cols = [c for c in og.columns if c != "Orthogroup"]
rows = []
for _, r in og.iterrows():
    votes = collections.Counter()
    for s in sp_cols:
        for g in [x.strip() for x in r[s].split(",") if x.strip()]:
            lbl = genelabel.get(g)
            if lbl: votes[lbl] += 1
    top, cnt = votes.most_common(1)[0] if votes else ("unannotated", 0)
    rows.append((r["Orthogroup"], top, cnt, sum(votes.values())))
ann = pd.DataFrame(rows, columns=["FamilyID","eggnog_consensus_label","top_votes","total_votes"])
ann.to_csv("eggnog/og_annotation_eggnog.tsv", sep="\t", index=False)
print(f"OGs with an eggNOG label: {(ann.eggnog_consensus_label!='unannotated').sum()} / {len(ann)}")
ann.head(8)

genes with an eggNOG label: 165752


OGs with an eggNOG label: 19542 / 19563


,FamilyID,eggnog_consensus_label,top_votes,total_votes
0,OG0000000,Vmn2r65,712,1509
1,OG0000001,nucleic acid binding,86,939
2,OG0000002,Zinc finger protein,444,828
3,OG0000003,Vmn2r65,468,567
4,OG0000004,Reverse transcriptase (RNA-dependent DNA polym...,64,321
5,OG0000005,Zinc finger protein,202,312
6,OG0000006,Vmn2r65,199,309
7,OG0000007,Vmn2r65,173,298


**Do the headline families (V2R / zinc-finger / MHC) still get called the same, under a fully independent labelling method?**
Compare the Hydrophis-only label to the eggNOG label per orthogroup, using a pattern that accounts for eggNOG's specific paralog naming (e.g. `Vmn2r65` for V2Rs).

In [3]:
hy = pd.read_csv("inputs/og_annotation.tsv", sep="\t").set_index("FamilyID")["consensus_product"]
eg = ann.set_index("FamilyID")["eggnog_consensus_label"]
df = pd.DataFrame({"hydrophis_label":hy, "eggnog_label":eg}).dropna()

def compare(hy_pattern, eg_pattern, name):
    hy_set = set(df.index[df.hydrophis_label.str.contains(hy_pattern, case=False, na=False, regex=True)])
    eg_set = set(df.index[df.eggnog_label.str.contains(eg_pattern, case=False, na=False, regex=True)])
    jac = len(hy_set & eg_set) / len(hy_set | eg_set) if (hy_set|eg_set) else 0
    print(f"{name:28s}: Hydrophis-only={len(hy_set):4d}  eggNOG={len(eg_set):4d}  "
          f"overlap={len(hy_set&eg_set):4d}  jaccard={jac:.3f}")

compare("Vomeronasal type-2", r"^Vmn2r", "V2R (paralog-aware)")
compare("zinc finger", "zinc finger", "Zinc finger")
compare("MHC|histocompatibility", "MHC|histocompatibility|HLA-|H2-|BOLA", "MHC (HLA-aware)")
compare("olfactory receptor|^OR[0-9]", "olfactory receptor|^OR[0-9]", "Olfactory receptor (control)")

V2R (paralog-aware)         : Hydrophis-only= 325  eggNOG= 167  overlap=  66  jaccard=0.155
Zinc finger                 : Hydrophis-only= 885  eggNOG= 131  overlap=  40  jaccard=0.041
MHC (HLA-aware)             : Hydrophis-only=  69  eggNOG=  34  overlap=  12  jaccard=0.132
Olfactory receptor (control): Hydrophis-only=  58  eggNOG= 187  overlap=  14  jaccard=0.061


```
**Interpretation:** individual-OG label agreement is low (jaccard ≈0.03–0.08) even under a proper orthology
tool, not just the earlier BLAST approximation. This rules out "BLAST was too crude" as the explanation. What's
happening: V2Rs, zinc-fingers and MHC are all **large, fast-duplicating gene families** — many near-identical
paralogues per genome. Both methods pick a *single* best match per gene, and with hundreds of near-identical
candidates on each side, small differences in which exact paralog is closest flip the "winning" label between
orthogroups constantly. This is a **labelling-resolution** problem specific to naming individual paralogues, not a
sign that the counts are unreliable — the count-based control needs no per-gene labels at all and
is unaffected.
```

In [4]:
import json
top20 = pd.read_csv("analysis/rapidly_evolving_families.tsv", sep="\t").head(20)["FamilyID"].tolist() \
    if os.path.exists("analysis/rapidly_evolving_families.tsv") else []
comp = pd.DataFrame({"FamilyID":top20,
    "hydrophis_label":[hy.get(o,"?") for o in top20],
    "eggnog_label":[eg.get(o,"?") for o in top20]})
comp.to_csv("eggnog/top20_rapidly_evolving_comparison.tsv", sep="\t", index=False)
comp

,FamilyID,hydrophis_label,eggnog_label
0,OG0000005,ZNF420: Zinc finger protein 420,Zinc finger protein
1,OG0000006,Vmn2r26: Vomeronasal type-2 receptor 26,Vmn2r65
2,OG0000007,Vmn2r26: Vomeronasal type-2 receptor 26,Vmn2r65
3,OG0000008,Vmn2r26: Vomeronasal type-2 receptor 26,K02A2.6-like
4,OG0000012,UBE2Z: Ubiquitin-conjugating enzyme E2 Z,Olfactory receptor
5,OG0000015,Vmn2r26: Vomeronasal type-2 receptor 26,Vmn2r65
6,OG0000017,"Class I histocompatibility antigen, F10 alpha ...",MR1
7,OG0000018,GTF2IRD2: General transcription factor II-I re...,Zinc finger protein
8,OG0000019,"DLA class II histocompatibility antigen, DR-1 ...",HLA-DPB1
9,OG0000020,Ig heavy chain V region 3,antigen binding


In [5]:
S = json.load(open("eggnog/summary.json"))
print(json.dumps(S, indent=2)[:800], "...")

{
  "path_used": "A (real eggNOG-mapper, diamond mode, eggNOG5 DB)",
  "why": "DB (61GB: eggnog.db + eggnog_proteins.dmnd) was re-fetched to /scratchdata1/.../prog/eggnog_db after the original Aug download was found deleted; eggnog conda env recreated; emapper.py run on all 12 proteomes.",
  "genes_annotated": {
    "caspera": 18251,
    "hcure": 18496,
    "hcurw": 18633,
    "hcy": 18451,
    "hmaj": 18921,
    "horn": 18512,
    "nhel": 18789,
    "nscut": 16370,
    "ptext": 16829,
    "tele": 18681,
    "vberus": 18511,
    "vkomodo": 17808
  },
  "total_genes_annotated": 218252,
  "ogs_relabeled": 19542,
  "ogs_total": 19563,
  "headline_agreement": "V2R (Vmn2r-pattern-aware) jaccard=0.053 (25/469 OGs agree); zinc-finger jaccard=0.027 (27/988); MHC/HLA jaccard=0.078 (7/90). Low indiv ...
